# MeteoPrep — exploration

Notebook d'exploration de l'étape 2 : récupération météo mensuelle par hôtel, renommage lisible et imputation.

Objectif : visualiser les entrées, les étapes intermédiaires (cache/API, renommage, grille année×mois) et remplir `../Output/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "MeteoPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Entrée — hôtels depuis RodPrep

In [ ]:
from meteo_prep.prep import MeteoPrep, READABLE_WEATHER

prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR)

if not (INPUT_DIR / "hotels.parquet").exists():
    if not (ROD_OUTPUT / "hotel_lookup.parquet").exists():
        raise FileNotFoundError("Exécuter d'abord RodPrep/Explore/explore.ipynb")
    hotels_path = prep.fill_input_from_rod(ROD_OUTPUT)
    print("Entrée créée depuis RodPrep :", hotels_path)
else:
    print("Entrée existante :", INPUT_DIR / "hotels.parquet")

hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
hotels[["hotel_code", "hotel_name", "hotel_brand", "hotel_city", "hotel_lat", "hotel_lon"]].head(10)

## 2. Enrichissement — météo brute (feature store / API)

Pour chaque hôtel, `EnrichHotelService.enrich()` lit le cache ou calcule les features météo.

In [ ]:
enrich_summary = []
for _, hotel in hotels.iterrows():
    code = str(hotel.get("hotel_code", ""))
    name = str(hotel.get("hotel_name", code))
    city = str(hotel.get("hotel_city", ""))
    result = prep._enrich.enrich(hotel_name=name, city=city, hotel_id=code, force_refresh=False)
    weather = result.features.weather_monthly or {}
    enrich_summary.append({
        "hotel_code": code,
        "hotel_name": name,
        "source": result.source,
        "nb_cles_meteo": len(weather),
        "warnings": "; ".join(result.warnings) if result.warnings else "",
    })

enrich_df = pd.DataFrame(enrich_summary)
print(f"Enrichissements : {len(enrich_df)} hôtels")
enrich_df

## 3. Aperçu clés brutes — premier hôtel

In [ ]:
sample_hotel = hotels.iloc[0]
code = str(sample_hotel.get("hotel_code", ""))
name = str(sample_hotel.get("hotel_name", code))
city = str(sample_hotel.get("hotel_city", ""))

sample_result = prep._enrich.enrich(hotel_name=name, city=city, hotel_id=code, force_refresh=False)
raw_weather = sample_result.features.weather_monthly or {}
raw_keys = sorted(k for k in raw_weather if k.startswith("d_m"))

print(f"Hôtel {code} — {len(raw_keys)} clés d_m*")
pd.DataFrame({"cle_brute": raw_keys[:18], "valeur": [raw_weather[k] for k in raw_keys[:18]]})

## 4. Renommage `_readable_monthly`

Transformation `d_m{MM}_{metric}_{stat}` → `meteo_{libelle}_{stat}`.

In [ ]:
monthly_readable = prep._readable_monthly(raw_weather)
readable_rows = []
for month, metrics in sorted(monthly_readable.items()):
    for col, val in sorted(metrics.items()):
        readable_rows.append({"mois": month, "colonne": col, "valeur": val})

readable_preview = pd.DataFrame(readable_rows)
print(f"Colonnes lisibles pour {code} : {readable_preview['colonne'].nunique()}")
readable_preview.head(18)

## 5. Grille `hotel_code × annee × mois` (2024–2025)

Une ligne par combinaison ; `_month_vector` injecte les métriques du mois.

In [ ]:
rows = []
for _, hotel in hotels.iterrows():
    hcode = str(hotel.get("hotel_code", ""))
    hname = str(hotel.get("hotel_name", hcode))
    city = str(hotel.get("hotel_city", ""))
    result = prep._enrich.enrich(hotel_name=hname, city=city, hotel_id=hcode, force_refresh=False)
    monthly = prep._readable_monthly(result.features.weather_monthly)
    for year in MeteoPrep.TARGET_YEARS:
        for month in range(1, 13):
            row = {"hotel_code": hcode, "hotel_name": hname, "annee": year, "mois": month}
            row.update(prep._month_vector(monthly, month))
            rows.append(row)

expanded = pd.DataFrame(rows)
print(f"Grille avant imputation : {expanded.shape[0]} lignes × {expanded.shape[1]} colonnes")
expanded.sort_values(["hotel_code", "annee", "mois"]).head(12)

## 6. Imputation `_impute_missing`

Par `(hotel_code, annee)` : ffill → bfill → moyenne annuelle → 0.

In [ ]:
meteo_cols = [c for c in expanded.columns if c.startswith("meteo_")]
missing_before = expanded[meteo_cols].isna().sum().sum() if meteo_cols else 0

imputed = prep._impute_missing(expanded)
missing_after = imputed[meteo_cols].isna().sum().sum() if meteo_cols else 0

print(f"NaN meteo avant imputation : {missing_before}")
print(f"NaN meteo après imputation : {missing_after}")
imputed.sort_values(["hotel_code", "annee", "mois"]).head(12)

## 7. Persistance `Output/`

In [ ]:
meteo_final = prep.run()
print(f"meteo_monthly : {meteo_final.shape}")
meteo_final.head()

print("\nFichiers produits :")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" ", path.name)